# LIFE on Google Colab — GossipCop++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **GossipCop++** only (~8253 LLM-pair articles: 4084 fake + 4169 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [1]:
# Confirm a GPU is attached
!nvidia-smi

Thu Jun 18 03:35:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/GossipCop++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top15.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('GossipCop++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
GossipCop++ found: True


In [4]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.7/644.7 kB 40.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 147.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.42.0 requires rich<14,>=12.4.4, but you have rich 11.2.0 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 11.2.0 which is incompatible.


In [5]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [ ]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
# from huggingface_hub import login
# login()

## Step 0 — Convert GossipCop++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (4084, fake) and `MR_true.jsonl` (4169, real) — the paper's binary task. HF/HR (human-written) are not used.

In [ ]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

## Step 1 — Key-sentence extraction (top-15)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-15** most impactful sentences per article (paper's k for GossipCop++). This is the slowest step (a forward pass per sentence per article).

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 15 --model_path "{BERT_CKPT}" --gpu 0

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [ ]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

## Step 4A — train the classifier (released head: BMES tags + CRF + majority vote)
Splits `FEATURES_LLAMA` into train/test (seed-0, deterministic) and trains the released Transformer classifier for **50 epochs** on the binary MF-vs-MR task. This head diverges from the paper — it tags tokens with B/M/E/S labels, CRF-decodes them, and recovers the article label by majority vote — so it is the **A-side** of the head A/B. Step 4B below is the paper-faithful head. Paper target for GossipCop++: **Acc 0.937 / F1 0.924**.

In [ ]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 10

## Step 4B — paper head (sigmoid + BCE, Eq 11–12)
Same CNN→Transformer trunk, but the head matches the paper: masked mean-pool → one sigmoid probability per article, trained with **binary cross-entropy** (fake=1, real=0) and evaluated directly at article level — no BMES tags, no CRF, no majority vote. Files: `LIFE_train/model_bce.py` + `LIFE_train/train_bce.py` (the originals are untouched and remain the A-side).

**The A/B is fair:** 4A and 4B consume the *same* `FEATURES_LLAMA` and the *same* seed-0 train/test split (`--split_dataset` here regenerates the identical split, so running 4A first is not required). The test set is only ~46 articles (≈2 accuracy points per article), so **re-run this cell with `--seed 1`, `2`, `3`, `4` and report mean ± std**. A-side references: 85.5/80.8 and 83.9/78.2; paper target 90.0/88.2. Checkpoint: `bce_en.pt`.

In [ ]:
#!python LIFE_train/train_bce.py \
#  --split_dataset \
#  --data_path "{FEATURES_LLAMA}" \
#  --train_path "{TRAIN_PATH}" \
#  --test_path "{TEST_PATH}" \
#  --num_train_epochs 10 \
#  --seed 0

## Multiclass experiment — 4-class HF / HR / MF / MR (LLaMA2-7B, released CRF/BMES head)

A separate, exploratory run that classifies all **four** PolitiFact++ categories
(human-fake, human-true, gpt3.5-fake, gpt3.5-true) instead of the paper's binary MF-vs-MR.
It reuses the released CRF/BMES head via `LIFE_train/train_multi.py` (a copy of `train.py`
with `en_labels` set to the four classes; `model.py` / `dataloader.py` are imported unchanged).

This needs its own LLaMA2-7B features (the binary `FEATURES_LLAMA` only has MF/MR), so
Steps 0m–3m re-run the pipeline with `--subset all` into separate `*_multi` paths — nothing
above is overwritten. Reference: an earlier 4-class run with **gpt2** features scored ~51.7%;
this swaps in the LLaMA2-7B features.

In [6]:
# --- 4-class (HF/HR/MF/MR) experiment paths (separate from the binary run above) ---
OUTPUT_MULTI     = f'{PROJECT_DIR}/dataset/output_multi'
KEY_SENT_MULTI   = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_multi_top15.jsonl'
BERT_CKPT_MULTI  = f'{PROJECT_DIR}/dataset/bert_multi.pt'
FEATURES_MULTI   = f'{PROJECT_DIR}/dataset/features_llama_multi'
TRAIN_PATH_MULTI = f'{PROJECT_DIR}/dataset/train_multi.jsonl'
TEST_PATH_MULTI  = f'{PROJECT_DIR}/dataset/test_multi.jsonl'

### Step 0m — Convert all four categories
`--subset all` emits `HF_fake` / `MF_fake` / `HR_true` / `MR_true` JSONL (~20505 articles).

In [7]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_MULTI}" --subset all

HF.json -> HF_fake.jsonl: 4084 records (label=human_fake)
MF.json -> MF_fake.jsonl: 4084 records (label=gpt3.5_fake)
HR.json -> HR_true.jsonl: 8168 records (label=human_true)
MR.json -> MR_true.jsonl: 4169 records (label=gpt3.5_true)


### Step 1m — Key-sentence extraction (top-15)
Trains a fresh binary BERT (fake = HF+MF, true = HR+MR) and keeps the top-15 sentences per
article. Slowest step; now over ~20505 articles.

In [8]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_MULTI}" --output_file "{KEY_SENT_MULTI}" --top_k 15 --model_path "{BERT_CKPT_MULTI}" --gpu 0

tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 192kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 771kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.26MB/s]
config.json: 100% 570/570 [00:00<00:00, 3.00MB/s]
model.safetensors: 100% 440M/440M [00:03<00:00, 143MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6077.55it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 


### Step 2m — Concatenate key sentences
Adds the `sentence` field to the records in `OUTPUT_MULTI` **in place** — re-run Step 0m to reset.

In [9]:
!python dataset/2_concate.py --folder_path "{OUTPUT_MULTI}" --important_sentences_file "{KEY_SENT_MULTI}"

Traceback (most recent call last):
  File "/content/drive/MyDrive/LIFE/dataset/2_concate.py", line 18, in <module>
    with open(important_sentences_file, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/LIFE/dataset/keySentence/important_sentences_multi_top15.jsonl'


### Step 3m — LLaMA2-7B reconstruction features
Same as the binary Step 3 but over all four files → `FEATURES_MULTI`.

In [10]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_MULTI}" --output_dir "{FEATURES_MULTI}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/__init__.py", line 16, in <module>
    from huggingface_hub.errors import (
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py", line 5, in <module>
    from httpx import HTTPError, Response
  File "/usr/local/lib/python3.12/dist-packages/httpx/__init__.py", line 15, in <module>
    from ._main import main
  File "/usr/local/lib/python3.12/dist-packages/httpx/_main.py", line 11, in <module>
    import rich.console
  File "/usr/local/lib/python3.12/dist-packages/rich/console.py", line 55, in <module>
    from .pretty import Pretty, is_expandable
  File "/usr/local/lib

### Step 4m — Train the 4-class classifier
Released CRF/BMES head over 16 BMES tags (4 classes × B/M/E/S), recovered to a 4-class label
by sentence majority vote. Prints Accuracy / Macro-F1 / per-class precision-recall in the
class id order printed at startup. Writes `linear_multi_en.pt`.

In [11]:
!python LIFE_train/train_multi.py --split_dataset --data_path "{FEATURES_MULTI}" --train_path "{TRAIN_PATH_MULTI}" --test_path "{TEST_PATH_MULTI}" --model Transformer --num_train_epochs 10

Traceback (most recent call last):
  File "/content/drive/MyDrive/LIFE/LIFE_train/train_multi.py", line 24, in <module>
    from sklearn.metrics import precision_score, recall_score
  File "/usr/local/lib/python3.12/dist-packages/sklearn/__init__.py", line 73, in <module>
    from .base import clone  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 19, in <module>
    from .utils._estimator_html_repr import _HTMLDocumentationLinkMixin, estimator_html_repr
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/__init__.py", line 15, in <module>
    from ._chunking import gen_batches, gen_even_slices
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_chunking.py", line 11, in <module>
    from ._param_validation import Interval, validate_params
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py", line 17, in <module>
    from .validation import _is_arraylike_not_scalar
  File

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.